# Vicmap FOI Exploration

This notebook explores the downloaded Vicmap FOI centroid snapshot for Iteration 1.

It does not modify the raw file or create the final processed dataset.

## 0.Download the raw GeoJSON snapshot from Vicmap API

In [4]:
from pathlib import Path
import requests
import time

output_dir = Path("../../data/raw/vicmap")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "foi_index_centroid_full_2026-08-26.geojson"

base_url = "https://opendata.maps.vic.gov.au/geoserver/wfs"

params = {
    "service": "WFS",
    "version": "1.0.0",
    "request": "GetFeature",
    "typeName": "open-data-platform:foi_index_centroid",
    "outputFormat": "application/json",
    "srsName": "EPSG:4326",
    "maxFeatures": 5000,
}

all_features = []
start_index = 0

while True:
    params["startIndex"] = start_index

    response = requests.get(base_url, params=params, timeout=120)
    response.raise_for_status()

    page = response.json()
    features = page.get("features", [])

    all_features.extend(features)

    print(f"Downloaded {len(all_features):,} records")

    if len(features) < 5000:
        break

    start_index += 5000
    time.sleep(1)

full_geojson = {
    "type": "FeatureCollection",
    "features": all_features,
    "crs": {
        "type": "name",
        "properties": {"name": "EPSG:4326"}
    }
}

import json

with output_path.open("w", encoding="utf-8") as file:
    json.dump(full_geojson, file)

print(f"Saved to: {output_path.resolve()}")
print(f"Total records: {len(all_features):,}")

Downloaded 5,000 records
Downloaded 10,000 records
Downloaded 15,000 records
Downloaded 20,000 records
Downloaded 25,000 records
Downloaded 30,000 records
Downloaded 35,000 records
Downloaded 40,000 records
Downloaded 45,000 records
Downloaded 50,000 records
Downloaded 55,000 records
Downloaded 60,000 records
Downloaded 65,000 records
Downloaded 70,000 records
Downloaded 75,000 records
Downloaded 80,000 records
Downloaded 85,000 records
Downloaded 90,000 records
Downloaded 95,000 records
Downloaded 100,000 records
Downloaded 105,000 records
Downloaded 106,084 records
Saved to: E:\Codex_work\active-together\data\raw\vicmap\foi_index_centroid_full_2026-08-26.geojson
Total records: 106,084


## 1. Load the raw GeoJSON snapshot

In [5]:
from pathlib import Path
import json
import pandas as pd

filename = 'foi_index_centroid_full_2026-08-26.geojson'
candidates = [
    Path('data/raw/vicmap') / filename,
    Path('../../data/raw/vicmap') / filename,
]
raw_path = next(path for path in candidates if path.exists())

with raw_path.open(encoding='utf-8') as file:
    geojson = json.load(file)

features = geojson['features']
properties = pd.DataFrame([feature['properties'] for feature in features])

print(f'File: {raw_path.resolve()}')
print(f'GeoJSON type: {geojson.get("type")}')
print(f'Records loaded: {len(features):,}')
print(f'CRS: {geojson.get("crs", {}).get("properties", {}).get("name", "not provided")}')
properties.head()

File: E:\Codex_work\active-together\data\raw\vicmap\foi_index_centroid_full_2026-08-26.geojson
GeoJSON type: FeatureCollection
Records loaded: 106,084
CRS: EPSG:4326


,ufi,pfi,feature_id,parent_feature_id,feature_type,feature_subtype,feature_status,name,name_label,parent_name,...,theme2,state,x_coord,y_coord,z_coord,create_date_pfi,superceded_pfi,feature_ufi,feature_create_date_ufi,create_date_ufi
0,35500808,5002,5002,NaN,admin facility,customer service centre,None,NaN,NaN,NaN,...,None,VIC,146.003833,-36.008278,None,2009-05-20T17:30:45Z,None,35500808,2009-10-13T07:33:59Z,2009-10-13T07:33:59Z
1,46173642,999506,999506,NaN,admin facility,customer service centre,None,YARRA RANGES COMMUNITY LINK CENTRE - UPWEY,Yarra Ranges Community Link Centre - Upwey,NaN,...,None,VIC,145.330734,-37.903747,None,2014-04-15T14:45:41Z,None,46173642,2014-04-16T06:59:41Z,2014-04-16T06:59:41Z
2,45880018,997364,997364,NaN,admin facility,customer service centre,None,COLAC-OTWAY APOLLO BAY CUSTOMER SERVICE CENTRE,Colac-Otway Apollo Bay Customer Service Centre,NaN,...,None,VIC,143.664253,-38.758977,None,2014-01-10T08:49:02Z,None,45880018,2014-01-10T22:07:33Z,2014-01-10T22:07:33Z
3,45995377,998157,998157,NaN,admin facility,customer service centre,None,CASTERTON CUSTOMER SERVICE CENTRE,Casterton Customer Service Centre,NaN,...,None,VIC,141.405403,-37.584939,None,2014-02-10T15:04:45Z,None,45995377,2014-02-11T06:58:36Z,2014-02-11T06:58:36Z
4,46757147,998297,998297,NaN,admin facility,customer service centre,None,MORNINGTON PENINSULA SHIRE COUNCIL - HASTINGS ...,Mornington Peninsula Shire Council - Hastings ...,NaN,...,None,VIC,145.194775,-38.308254,None,2014-02-20T15:41:15Z,None,46757147,2014-10-22T07:34:10Z,2014-10-22T07:34:10Z


## 2. Inspect fields and missing values

In [6]:
field_summary = pd.DataFrame({
    'dtype': properties.dtypes.astype(str),
    'missing_count': properties.isna().sum(),
    'missing_percent': (properties.isna().mean() * 100).round(2),
    'unique_count': properties.nunique(dropna=True),
}).sort_values('missing_percent', ascending=False)

field_summary

,dtype,missing_count,missing_percent,unique_count
feature_status,object,106084,100.00,0
superceded_pfi,object,106084,100.00,0
theme1,object,106084,100.00,0
theme2,object,106084,100.00,0
z_coord,object,106084,100.00,0
child_exists,str,95923,90.42,1
parent_name,str,82643,77.90,6711
parent_feature_id,float64,78284,73.79,10143
auth_org_code,str,64073,60.40,106
auth_org_verified,str,63607,59.96,338


## 3. Explore feature categories

In [7]:
for column in ['feature_type', 'feature_subtype', 'feature_status']:
    print(f'### {column}')
    display(properties[column].value_counts(dropna=False).head(30).to_frame('count'))
    print()

### feature_type


,count
feature_type,
reserve,28508
sport facility,13024
recreational resource,10161
education centre,6197
care facility,6087
excavation site,4982
sign,4905
storage facility,4335
landmark,3728



### feature_subtype


,count
feature_subtype,
park,26596
playground,6081
sports ground,5797
child care,5119
emergency marker,4817
education complex,2656
silo,2581
church,2218
tennis court,1959



### feature_status


,count
feature_status,
None,106084


## 4. Inspect names and coordinate quality

In [8]:
name_columns = ['name', 'name_label', 'parent_name']
display(properties[name_columns].head(20))

coordinate_summary = pd.DataFrame({
    'x_coord': pd.to_numeric(properties['x_coord'], errors='coerce'),
    'y_coord': pd.to_numeric(properties['y_coord'], errors='coerce'),
}).describe()
coordinate_summary

,name,name_label,parent_name
0,NaN,NaN,NaN
1,YARRA RANGES COMMUNITY LINK CENTRE - UPWEY,Yarra Ranges Community Link Centre - Upwey,NaN
2,COLAC-OTWAY APOLLO BAY CUSTOMER SERVICE CENTRE,Colac-Otway Apollo Bay Customer Service Centre,NaN
3,CASTERTON CUSTOMER SERVICE CENTRE,Casterton Customer Service Centre,NaN
4,MORNINGTON PENINSULA SHIRE COUNCIL - HASTINGS ...,Mornington Peninsula Shire Council - Hastings ...,NaN
5,MILDURA RURAL CITY COUNCIL DEAKIN AVENUE SERVI...,Mildura Rural City Council Deakin Avenue Servi...,NaN
6,RURAL CITY OF MILDURA OUYEN SERVICE CENTRE,Rural City Of Mildura Ouyen Service Centre,NaN
7,SHIRE OF BULOKE BIRCHIP CUSTOMER SRVICE CENTRE,Shire Of Buloke Birchip Customer Srvice Centre,NaN
8,RESORT ENTRY GATE,Resort Entry Gate,NaN
9,GREATER DANDENONG CUSTOMER SERVICE CENTRE,Greater Dandenong Customer Service Centre,NaN


,x_coord,y_coord
count,106084.000000,106084.000000
mean,144.865247,-37.395895
std,1.551016,0.918116
min,139.810235,-39.129499
25%,144.279342,-37.981048
50%,145.003539,-37.741280
75%,145.364786,-36.880790
max,150.049187,-33.018440


In [9]:
print('Missing names:')
print(properties['name_label'].isna().sum())
print('Missing coordinates:')
print(properties[['x_coord', 'y_coord']].isna().any(axis=1).sum())
print('Potential duplicate feature IDs:')
print(properties['feature_id'].duplicated().sum())

invalid_longitude = ~properties['x_coord'].between(140, 150, inclusive='both')
invalid_latitude = ~properties['y_coord'].between(-40, -34, inclusive='both')
print(f'Coordinates outside a broad Victoria range: {(invalid_longitude | invalid_latitude).sum()}')

Missing names:
49224
Missing coordinates:
0
Potential duplicate feature IDs:
0
Coordinates outside a broad Victoria range: 104


## Initial observations

Record findings here after running the cells. Do not decide final categories or exclusion rules until the distributions and name patterns have been reviewed.

## Initial observations

- The downloaded Vicmap FOI snapshot contains 106,084 point features in GeoJSON format using CRS EPSG:4326.
- The dataset is a broad feature-of-interest dataset rather than a purpose-built activity-place dataset. It contains 30 `feature_type` values and 180 `feature_subtype` values.
- The most relevant potential activity categories include parks, playgrounds, sports grounds, tennis courts, reserves, camp grounds and other recreational resources.
- The dataset also contains many clearly irrelevant categories, including administrative facilities, hospitals, industrial facilities, emergency facilities, storage facilities and control points. These will require exclusion rules before the data can be used for recommendations.
- `feature_id` values are unique and no coordinates are missing, which provides a useful basis for spatial filtering.
- Name information is incomplete: `name` and `name_label` are missing for approximately 46.4% of records. Classification should therefore use `feature_type` and `feature_subtype` in addition to name fields.
- Several fields are completely empty, including `feature_status`, `superceded_pfi`, `theme1`, `theme2` and `z_coord`. These fields are unlikely to be useful in the processed dataset.
- The data includes records from VIC, NSW and SA. Since the project focuses on Greater Melbourne, a spatial filter will be required rather than relying only on the `state` field.
- The initial coordinate-range check identified 104 records outside an arbitrary broad range. This does not necessarily indicate invalid coordinates, because the selected range was narrower than the full source extent. A validated Greater Melbourne boundary should be used for spatial filtering.
- The next exploration step is to review the complete `feature_type` and `feature_subtype` distributions and identify candidate inclusion and exclusion categories.

## 5. Feature type and subtype inventory

This section examines the relationship between `feature_type` and `feature_subtype` before defining any inclusion or exclusion rules.

In [10]:
category_inventory = (
    properties
    .groupby(["feature_type", "feature_subtype"], dropna=False)
    .agg(
        record_count=("feature_id", "size"),
        named_records=("name_label", "count")
    )
    .reset_index()
)

category_inventory["missing_name_count"] = (
    category_inventory["record_count"]
    - category_inventory["named_records"]
)

category_inventory["missing_name_percent"] = (
    category_inventory["missing_name_count"]
    / category_inventory["record_count"]
    * 100
).round(2)

category_inventory = category_inventory.sort_values(
    ["feature_type", "record_count"],
    ascending=[True, False]
)

category_inventory

,feature_type,feature_subtype,record_count,named_records,missing_name_count,missing_name_percent
4,admin facility,office,319,274,45,14.11
8,admin facility,tourist information centre,148,143,5,3.38
3,admin facility,municipal office,144,140,4,2.78
2,admin facility,law court,87,86,1,1.15
1,admin facility,justice service,47,47,0,0.00
...,...,...,...,...,...,...
172,storage facility,depot,289,143,146,50.52
179,storage facility,stockyard,60,38,22,36.67
173,storage facility,gas tank,25,1,24,96.00
177,storage facility,petroleum tank,17,3,14,82.35


In [11]:
def show_subtypes(feature_type):
    result = category_inventory[
        category_inventory["feature_type"] == feature_type
    ]

    return result.reset_index(drop=True)


show_subtypes("reserve")

,feature_type,feature_subtype,record_count,named_records,missing_name_count,missing_name_percent
0,reserve,park,26596,12863,13733,51.64
1,reserve,cemetery,854,807,47,5.50
2,reserve,conservation park,734,729,5,0.68
3,reserve,gardens,183,166,17,9.29
4,reserve,national park,113,113,0,0.00
5,reserve,amusement centre,14,10,4,28.57
6,reserve,zoo,8,8,0,0.00
7,reserve,city square,6,6,0,0.00


In [12]:
show_subtypes("sport facility")

,feature_type,feature_subtype,record_count,named_records,missing_name_count,missing_name_percent
0,sport facility,sports ground,5797,889,4908,84.66
1,sport facility,tennis court,1959,309,1650,84.23
2,sport facility,training track,1101,12,1089,98.91
3,sport facility,sports complex,811,488,323,39.83
4,sport facility,netball court,594,37,557,93.77
5,sport facility,bowling green,583,491,92,15.78
6,sport facility,golf course,468,429,39,8.33
7,sport facility,swimming pool,355,95,260,73.24
8,sport facility,basketball court,242,102,140,57.85
9,sport facility,motor track,152,88,64,42.11


In [13]:
show_subtypes("recreational resource")

,feature_type,feature_subtype,record_count,named_records,missing_name_count,missing_name_percent
0,recreational resource,playground,6081,850,5231,86.02
1,recreational resource,club house,1839,633,1206,65.58
2,recreational resource,picnic site,835,473,362,43.35
3,recreational resource,day visitor area,401,401,0,0.00
4,recreational resource,skate park,328,297,31,9.45
5,recreational resource,rotunda,181,31,150,82.87
6,recreational resource,group camp,126,120,6,4.76
7,recreational resource,trailhead,117,117,0,0.00
8,recreational resource,bmx track,113,52,61,53.98
9,recreational resource,hut,104,101,3,2.88


In [14]:
show_subtypes("community space")

,feature_type,feature_subtype,record_count,named_records,missing_name_count,missing_name_percent
0,community space,camp ground,1676,1628,48,2.86
1,community space,caravan park,611,601,10,1.64
2,community space,rest area,558,227,331,59.32
3,community space,parking area,394,282,112,28.43
4,community space,showground,80,69,11,13.75


In [15]:
show_subtypes("community venue")

,feature_type,feature_subtype,record_count,named_records,missing_name_count,missing_name_percent
0,community venue,hall,1804,1586,218,12.08
1,community venue,community centre,454,411,43,9.47
2,community venue,senior citizens,182,171,11,6.04
3,community venue,neighbourhood house,72,72,0,0.00


In [16]:
show_subtypes("reserve")

,feature_type,feature_subtype,record_count,named_records,missing_name_count,missing_name_percent
0,reserve,park,26596,12863,13733,51.64
1,reserve,cemetery,854,807,47,5.50
2,reserve,conservation park,734,729,5,0.68
3,reserve,gardens,183,166,17,9.29
4,reserve,national park,113,113,0,0.00
5,reserve,amusement centre,14,10,4,28.57
6,reserve,zoo,8,8,0,0.00
7,reserve,city square,6,6,0,0.00


## 6. Create a subtype review table

The dataset contains 180 feature subtypes. Before filtering individual records, we will create a reusable review table that assigns each subtype an initial decision: `include`, `exclude`, or `review`.

The table will also record a proposed activity category, the basis for classification confidence, and any relevant notes. These decisions are made at the subtype level rather than manually reviewing all 106,084 records.

This step does not modify the raw GeoJSON. During the wrangling stage, the completed review table will be joined to the full dataset using `feature_type` and `feature_subtype`. Ambiguous records may then be reviewed using `name_label` or other available context.

In [18]:
# Create a review table from the subtype inventory
subtype_review = category_inventory.copy()

# Add review and classification columns
subtype_review["decision"] = "review"
subtype_review["activity_category"] = pd.NA
subtype_review["confidence_basis"] = pd.NA
subtype_review["notes"] = pd.NA

# Arrange columns in a readable order
subtype_review = subtype_review[
    [
        "feature_type",
        "feature_subtype",
        "record_count",
        "named_records",
        "missing_name_count",
        "missing_name_percent",
        "decision",
        "activity_category",
        "confidence_basis",
        "notes",
    ]
]

# Preview the first 20 subtype rules
subtype_review.head(20)

,feature_type,feature_subtype,record_count,named_records,missing_name_count,missing_name_percent,decision,activity_category,confidence_basis,notes
4,admin facility,office,319,274,45,14.11,review,<NA>,<NA>,<NA>
8,admin facility,tourist information centre,148,143,5,3.38,review,<NA>,<NA>,<NA>
3,admin facility,municipal office,144,140,4,2.78,review,<NA>,<NA>,<NA>
2,admin facility,law court,87,86,1,1.15,review,<NA>,<NA>,<NA>
1,admin facility,justice service,47,47,0,0.00,review,<NA>,<NA>,<NA>
0,admin facility,customer service centre,35,29,6,17.14,review,<NA>,<NA>,<NA>
6,admin facility,prison complex,20,1,19,95.00,review,<NA>,<NA>,<NA>
5,admin facility,prison,17,17,0,0.00,review,<NA>,<NA>,<NA>
7,admin facility,research station,12,12,0,0.00,review,<NA>,<NA>,<NA>
10,admin facility,youth justice,3,3,0,0.00,review,<NA>,<NA>,<NA>


In [19]:
# Update the review decision for one feature subtype
def set_subtype_rule(
    feature_type,
    feature_subtype,
    decision,
    activity_category=None,
    confidence_basis=None,
    notes=None,
):
    # Locate the matching feature type and subtype
    mask = (
        (subtype_review["feature_type"] == feature_type)
        & (subtype_review["feature_subtype"] == feature_subtype)
    )

    # Assign the proposed classification values
    subtype_review.loc[mask, "decision"] = decision
    subtype_review.loc[mask, "activity_category"] = activity_category
    subtype_review.loc[mask, "confidence_basis"] = confidence_basis
    subtype_review.loc[mask, "notes"] = notes

In [20]:
# Include parks as suitable open-space destinations
set_subtype_rule(
    feature_type="reserve",
    feature_subtype="park",
    decision="include",
    activity_category="park",
    confidence_basis="high: explicit subtype",
    notes="Suitable open-space activity destination",
)

# Exclude cemeteries from activity recommendations
set_subtype_rule(
    feature_type="reserve",
    feature_subtype="cemetery",
    decision="exclude",
    confidence_basis="high: explicit subtype",
    notes="Not suitable as a family activity recommendation",
)

# Include playgrounds even when a place name is unavailable
set_subtype_rule(
    feature_type="recreational resource",
    feature_subtype="playground",
    decision="include",
    activity_category="playground",
    confidence_basis="high: explicit subtype",
    notes="Name is not required for initial classification",
)

# Keep ambiguous training tracks for further review
set_subtype_rule(
    feature_type="sport facility",
    feature_subtype="training track",
    decision="review",
    confidence_basis="low: ambiguous subtype",
    notes="Review name_label where available",
)

In [21]:
# Display subtype rules that are no longer marked for review
subtype_review[
    subtype_review["decision"] != "review"
]

,feature_type,feature_subtype,record_count,named_records,missing_name_count,missing_name_percent,decision,activity_category,confidence_basis,notes
132,recreational resource,playground,6081,850,5231,86.02,include,playground,high: explicit subtype,Name is not required for initial classification
142,reserve,park,26596,12863,13733,51.64,include,park,high: explicit subtype,Suitable open-space activity destination
137,reserve,cemetery,854,807,47,5.50,exclude,None,high: explicit subtype,Not suitable as a family activity recommendation


In [22]:
# Count the current include, exclude and review decisions
subtype_review["decision"].value_counts(dropna=False)

decision
review     177
include      2
exclude      1
Name: count, dtype: int64

## 7. Apply the initial seven-category mapping

Seven clear activity categories are used for the initial classification. Only subtypes with an explicit and sufficiently reliable meaning are assigned at this stage. Ambiguous, access-dependent or potentially paid facilities remain under review.

The seven categories are:

- `playground`
- `park_and_garden`
- `sports_ground`
- `court`
- `trail_access`
- `skate_bmx`
- `picnic_day_use`

In [23]:
# Define the seven permitted activity categories
allowed_activity_categories = {
    "playground",
    "park_and_garden",
    "sports_ground",
    "court",
    "trail_access",
    "skate_bmx",
    "picnic_day_use",
}

# Map explicit subtypes to high-confidence activity categories
include_rules = [
    ("recreational resource", "playground", "playground"),
    ("reserve", "park", "park_and_garden"),
    ("reserve", "gardens", "park_and_garden"),

    ("sport facility", "sports ground", "sports_ground"),
    ("sport facility", "sports complex", "sports_ground"),
    ("sport facility", "athletic field", "sports_ground"),
    ("sport facility", "baseball field", "sports_ground"),
    ("sport facility", "hockey ground", "sports_ground"),

    ("sport facility", "tennis court", "court"),
    ("sport facility", "basketball court", "court"),
    ("sport facility", "netball court", "court"),

    ("recreational resource", "trailhead", "trail_access"),

    ("recreational resource", "skate park", "skate_bmx"),
    ("recreational resource", "bmx track", "skate_bmx"),

    ("recreational resource", "picnic site", "picnic_day_use"),
    ("recreational resource", "day visitor area", "picnic_day_use"),
]

# Apply each high-confidence inclusion rule
for feature_type, feature_subtype, activity_category in include_rules:
    set_subtype_rule(
        feature_type=feature_type,
        feature_subtype=feature_subtype,
        decision="include",
        activity_category=activity_category,
        confidence_basis="high: explicit subtype",
        notes=f"Mapped from explicit subtype: {feature_subtype}",
    )

In [24]:
# Find included rows with an invalid or missing category
invalid_included_categories = subtype_review[
    (subtype_review["decision"] == "include")
    & (~subtype_review["activity_category"].isin(
        allowed_activity_categories
    ))
]

# Stop execution if an invalid category is found
assert invalid_included_categories.empty, (
    "An included subtype has an invalid activity category."
)

print("All included subtypes use one of the seven approved categories.")

All included subtypes use one of the seven approved categories.


In [25]:
# Summarise the initial high-confidence category mapping
category_mapping_summary = (
    subtype_review[
        subtype_review["decision"] == "include"
    ]
    .groupby("activity_category")
    .agg(
        subtype_count=("feature_subtype", "count"),
        record_count=("record_count", "sum"),
    )
    .sort_values("record_count", ascending=False)
)

category_mapping_summary

,subtype_count,record_count
activity_category,,
park_and_garden,2,26779
sports_ground,5,6919
playground,1,6081
court,3,2795
picnic_day_use,2,1236
skate_bmx,2,441
trail_access,1,117


## 8. Apply high-confidence exclusion rules

Clearly unsuitable feature types and subtypes are excluded from activity-place recommendations. These rules cover administrative, industrial, emergency, infrastructure, residential and other non-activity locations.

Paid, access-dependent or ambiguous facilities are not excluded automatically at this stage. They remain under review.

In [26]:
# Define feature types that are clearly unrelated to activity discovery
excluded_feature_types = {
    "admin facility",
    "agricultural area",
    "care facility",
    "commercial facility",
    "communication service",
    "control point",
    "defence site",
    "dumping ground",
    "education centre",
    "emergency facility",
    "excavation site",
    "health facility",
    "hospital",
    "industrial facility",
    "pipeline",
    "pipeline facility",
    "place of worship",
    "power facility",
    "residential building",
    "sign",
    "storage facility",
}

# Exclude every subtype belonging to a clearly irrelevant feature type
for feature_type in excluded_feature_types:
    matching_subtypes = subtype_review.loc[
        subtype_review["feature_type"] == feature_type,
        "feature_subtype",
    ]

    for feature_subtype in matching_subtypes:
        set_subtype_rule(
            feature_type=feature_type,
            feature_subtype=feature_subtype,
            decision="exclude",
            activity_category=None,
            confidence_basis="high: irrelevant feature type",
            notes="Not an activity-place feature type",
        )

In [27]:
# Define unsuitable subtypes found inside otherwise mixed feature types
explicit_exclusion_rules = [
    ("reserve", "cemetery", "Not suitable as an activity destination"),
    ("community space", "parking area", "Transport infrastructure rather than an activity destination"),
    ("community space", "rest area", "Rest facility rather than an activity destination"),
    ("recreational resource", "club house", "Access may require club membership"),
    ("recreational resource", "grandstand", "Supporting infrastructure rather than an activity destination"),
    ("recreational resource", "rotunda", "Supporting structure rather than an activity destination"),
    ("recreational resource", "hut", "Supporting structure rather than an activity destination"),
    ("sport facility", "target range", "Unsuitable for general child activity recommendations"),
    ("sport facility", "motor track", "Unsuitable for casual family activity recommendations"),
    ("sport facility", "equestrian facility", "Access-dependent specialist facility"),
    ("sport facility", "racecourse", "Access-dependent specialist facility"),
    ("sport facility", "horse racetrack", "Access-dependent specialist facility"),
    ("sport facility", "harness racetrack", "Access-dependent specialist facility"),
    ("sport facility", "greyhound racetrack", "Unsuitable for child activity recommendations"),
]

# Apply each explicit subtype exclusion rule
for feature_type, feature_subtype, reason in explicit_exclusion_rules:
    set_subtype_rule(
        feature_type=feature_type,
        feature_subtype=feature_subtype,
        decision="exclude",
        activity_category=None,
        confidence_basis="high: explicit subtype",
        notes=reason,
    )

In [28]:
# Summarise the current review decisions
decision_summary = (
    subtype_review
    .groupby("decision")
    .agg(
        subtype_count=("feature_subtype", "count"),
        record_count=("record_count", "sum"),
    )
)

decision_summary

,subtype_count,record_count
decision,,
exclude,126,48378
include,16,44368
review,38,13338


## 9. Inspect the remaining review subtypes

The remaining subtypes cannot yet be included or excluded with sufficient confidence. This section lists them by record count and examines representative place names where available.

The review will help distinguish potentially suitable locations from paid, restricted, indoor or semantically ambiguous facilities.

In [29]:
# Select unresolved subtypes and order them by record count
remaining_review = (
    subtype_review[
        subtype_review["decision"] == "review"
    ]
    .sort_values("record_count", ascending=False)
    .reset_index(drop=True)
)

# Display all remaining review subtypes
remaining_review

,feature_type,feature_subtype,record_count,named_records,missing_name_count,missing_name_percent,decision,activity_category,confidence_basis,notes
0,community venue,hall,1804,1586,218,12.08,review,<NA>,<NA>,<NA>
1,landmark,tower,1754,327,1427,81.36,review,<NA>,<NA>,<NA>
2,community space,camp ground,1676,1628,48,2.86,review,<NA>,<NA>,<NA>
3,sport facility,training track,1101,12,1089,98.91,review,None,low: ambiguous subtype,Review name_label where available
4,landmark,monument,1008,926,82,8.13,review,<NA>,<NA>,<NA>
5,reserve,conservation park,734,729,5,0.68,review,<NA>,<NA>,<NA>
6,community space,caravan park,611,601,10,1.64,review,<NA>,<NA>,<NA>
7,sport facility,bowling green,583,491,92,15.78,review,<NA>,<NA>,<NA>
8,landmark,lookout,491,286,205,41.75,review,<NA>,<NA>,<NA>
9,sport facility,golf course,468,429,39,8.33,review,<NA>,<NA>,<NA>


In [30]:
# Display representative names for one subtype under review
def show_name_examples(feature_type, feature_subtype, sample_size=10):
    matching_records = properties[
        (properties["feature_type"] == feature_type)
        & (properties["feature_subtype"] == feature_subtype)
    ]

    named_examples = (
        matching_records["name_label"]
        .dropna()
        .drop_duplicates()
        .head(sample_size)
        .reset_index(drop=True)
    )

    print(f"Feature type: {feature_type}")
    print(f"Feature subtype: {feature_subtype}")
    print(f"Total records: {len(matching_records):,}")
    print(
        f"Records with names: "
        f"{matching_records['name_label'].notna().sum():,}"
    )

    return named_examples

## 10. Review findings and next classification steps

The initial exploration examined 106,084 Vicmap FOI records across 30 feature types and 180 feature subtypes. A subtype-level review table was created because broad feature types often contain both suitable and unsuitable locations.

Seven clear activity categories were defined:

- `playground`
- `park_and_garden`
- `sports_ground`
- `court`
- `trail_access`
- `skate_bmx`
- `picnic_day_use`

High-confidence subtype rules initially classified 16 subtypes as suitable and 126 as unsuitable. This accounted for 44,368 included records and 48,378 excluded records. The remaining 38 subtypes, representing 13,338 records, required further review.

Representative `name_label` values were examined for ambiguous subtypes. This showed that some subtypes, such as `training track`, `tourist attraction` and `conservation park`, contain locations with substantially different purposes. Other subtypes, including swimming pools, bowling greens, golf courses, zoos and amusement centres, may provide activities but have unverified costs, opening hours, membership requirements or indoor access conditions.

The next step is to extend the review decision from three to four possible outcomes:

- `include`: suitable for ranked activity recommendations
- `exclude`: unsuitable for the product’s activity recommendations
- `fallback`: potentially useful but affected by unverified access, cost or operating conditions
- `review`: insufficient evidence for a reliable decision

The remaining subtypes will then be assigned to one of these outcomes. Only included records will receive one of the seven approved activity categories. Ambiguous records will not be forced into a generic category.

This classification remains part of the exploration stage. The raw GeoJSON will not be modified. The completed subtype mapping will later be joined to the full dataset during wrangling.

## 11. Classify the remaining reviewed subtypes

The reviewed subtypes are now assigned using the representative name evidence and the product constraints.

Clearly unsuitable locations are excluded. Potential activity venues with unverified costs, opening hours, membership requirements or indoor access are assigned to `fallback`. Subtypes that remain semantically mixed or lack sufficient access information stay under `review`.

In [36]:
# Define reviewed subtypes that are unsuitable for ranked recommendations
additional_exclusion_rules = [
    ("community venue", "hall", "Building rather than a specific activity destination"),
    ("landmark", "tower", "Landmark rather than an accessible activity destination"),
    ("community space", "camp ground", "Overnight facility outside the short-activity use case"),
    ("landmark", "monument", "Landmark rather than an activity destination"),
    ("community space", "caravan park", "Accommodation facility outside the short-activity use case"),
    ("community venue", "community centre", "Activities and access cannot be inferred from the location alone"),
    ("cultural centre", "library", "Not a physical activity destination"),
    ("cultural centre", "museum", "Not a physical activity destination"),
    ("place", "historic site", "Does not directly identify a physical activity opportunity"),
    ("community venue", "senior citizens", "Not relevant to the target child activity use case"),
    ("recreational resource", "group camp", "Booking-dependent group facility"),
    ("cableway", "cableway terminal", "Transport infrastructure rather than an activity destination"),
    ("cableway", "chairlift", "Access-dependent transport infrastructure"),
    ("community space", "showground", "Event-dependent facility without verified public access"),
    ("cultural centre", "art gallery", "Not a physical activity destination"),
    ("community venue", "neighbourhood house", "Activities and access cannot be inferred from the location alone"),
    ("cableway", "t-bar tow", "Access-dependent specialist infrastructure"),
    ("sport facility", "boating club", "Membership and equipment-dependent facility"),
    ("cableway", "poma tow", "Access-dependent specialist infrastructure"),
    ("recreational resource", "amphitheatre", "Supporting venue rather than a physical activity destination"),
    ("cultural centre", "observatory", "Access-dependent venue rather than a casual activity location"),
    ("landmark", "cairn", "Landmark rather than a reliable activity destination"),
]

# Apply the additional high-confidence exclusion rules
for feature_type, feature_subtype, reason in additional_exclusion_rules:
    set_subtype_rule(
        feature_type=feature_type,
        feature_subtype=feature_subtype,
        decision="exclude",
        activity_category=None,
        confidence_basis="high: subtype and name evidence",
        notes=reason,
    )

In [37]:
# Define potential activity venues with unverified access conditions
fallback_rules = [
    ("sport facility", "bowling green", "Membership or booking requirements are unverified"),
    ("sport facility", "golf course", "Cost and public access are unverified"),
    ("sport facility", "swimming pool", "Cost, opening hours and indoor status are unverified"),
    ("sport facility", "croquet green", "Membership or booking requirements are unverified"),
    ("reserve", "amusement centre", "Cost and opening hours are unverified"),
    ("reserve", "zoo", "Cost and opening hours are unverified"),
    ("sport facility", "golf driving range", "Cost and public access are unverified"),
    ("cultural centre", "aquarium", "Cost, opening hours and indoor access are unverified"),
]

# Assign access-dependent venues to the unverified fallback group
for feature_type, feature_subtype, reason in fallback_rules:
    set_subtype_rule(
        feature_type=feature_type,
        feature_subtype=feature_subtype,
        decision="fallback",
        activity_category=None,
        confidence_basis="medium: explicit subtype, access unverified",
        notes=reason,
    )

In [38]:
# Define every permitted review decision
allowed_decisions = {
    "include",
    "exclude",
    "fallback",
    "review",
}

# Confirm that no unexpected decision value exists
invalid_decisions = subtype_review[
    ~subtype_review["decision"].isin(allowed_decisions)
]

assert invalid_decisions.empty, "Unexpected decision value found."

print("All subtype decisions are valid.")

All subtype decisions are valid.


In [39]:
# Summarise subtype and record counts for each decision
updated_decision_summary = (
    subtype_review
    .groupby("decision")
    .agg(
        subtype_count=("feature_subtype", "count"),
        record_count=("record_count", "sum"),
    )
)

updated_decision_summary

,subtype_count,record_count
decision,,
exclude,148,57258
fallback,8,1519
include,16,44368
review,8,2939


In [40]:
# Display the small set of subtypes that still require investigation
remaining_review = (
    subtype_review[
        subtype_review["decision"] == "review"
    ]
    .sort_values("record_count", ascending=False)
    .reset_index(drop=True)
)

remaining_review

,feature_type,feature_subtype,record_count,named_records,missing_name_count,missing_name_percent,decision,activity_category,confidence_basis,notes
0,sport facility,training track,1101,12,1089,98.91,review,None,low: ambiguous subtype,Review name_label where available
1,reserve,conservation park,734,729,5,0.68,review,<NA>,<NA>,<NA>
2,landmark,lookout,491,286,205,41.75,review,<NA>,<NA>,<NA>
3,landmark,tourist attraction,466,456,10,2.15,review,<NA>,<NA>,<NA>
4,reserve,national park,113,113,0,0.00,review,<NA>,<NA>,<NA>
5,sport facility,velodrome,27,6,21,77.78,review,<NA>,<NA>,<NA>
6,reserve,city square,6,6,0,0.00,review,<NA>,<NA>,<NA>
7,cultural centre,arboretum,1,1,0,0.00,review,<NA>,<NA>,<NA>


In [41]:
# Include the arboretum as an outdoor park-and-garden location
set_subtype_rule(
    feature_type="cultural centre",
    feature_subtype="arboretum",
    decision="include",
    activity_category="park_and_garden",
    confidence_basis="high: explicit subtype",
    notes="Outdoor garden environment suitable for walking and exploration",
)

In [42]:
# Confirm that the arboretum rule was applied
subtype_review[
    (subtype_review["feature_type"] == "cultural centre")
    & (subtype_review["feature_subtype"] == "arboretum")
]

,feature_type,feature_subtype,record_count,named_records,missing_name_count,missing_name_percent,decision,activity_category,confidence_basis,notes
41,cultural centre,arboretum,1,1,0,0.0,include,park_and_garden,high: explicit subtype,Outdoor garden environment suitable for walkin...


In [43]:
# Recalculate the decision summary after applying the rule
updated_decision_summary = (
    subtype_review
    .groupby("decision")
    .agg(
        subtype_count=("feature_subtype", "count"),
        record_count=("record_count", "sum"),
    )
)

updated_decision_summary

,subtype_count,record_count
decision,,
exclude,148,57258
fallback,8,1519
include,17,44369
review,7,2938


## 12. Remaining review strategy

Seven subtypes remain unresolved after the subtype-level classification. They are not forced into an activity category because their suitability depends on place names, public access, entry locations or supporting datasets.

`training track` and `tourist attraction` require record-level name analysis. `conservation park`, `national park` and `lookout` require additional access or route evidence. `velodrome` may be treated as an unverified fallback facility, while the small number of `city square` records can be reviewed manually.

These records will remain under `review` until the required evidence is available. Low-confidence records will not be included in ranked recommendations.

In [44]:
# Locate the repository root from the current notebook directory
project_root = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / ".git").exists()
)

# Create the tracked validation output directory
review_output_dir = (
    project_root
    / "data"
    / "validation"
    / "vicmap"
)

review_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

# Save the subtype review table for team review and later wrangling
review_output_path = (
    review_output_dir
    / "vicmap_subtype_review.csv"
)

subtype_review.to_csv(
    review_output_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"Saved review table to: {review_output_path}")

Saved review table to: e:\Codex_work\active-together\data\validation\vicmap\vicmap_subtype_review.csv


In [45]:
# Read the saved table back and verify its dimensions
saved_subtype_review = pd.read_csv(review_output_path)

print(f"Saved rows: {len(saved_subtype_review)}")
print(f"Saved columns: {len(saved_subtype_review.columns)}")

saved_subtype_review.head()

Saved rows: 180
Saved columns: 10


,feature_type,feature_subtype,record_count,named_records,missing_name_count,missing_name_percent,decision,activity_category,confidence_basis,notes
0,admin facility,office,319,274,45,14.11,exclude,NaN,high: irrelevant feature type,Not an activity-place feature type
1,admin facility,tourist information centre,148,143,5,3.38,exclude,NaN,high: irrelevant feature type,Not an activity-place feature type
2,admin facility,municipal office,144,140,4,2.78,exclude,NaN,high: irrelevant feature type,Not an activity-place feature type
3,admin facility,law court,87,86,1,1.15,exclude,NaN,high: irrelevant feature type,Not an activity-place feature type
4,admin facility,justice service,47,47,0,0.00,exclude,NaN,high: irrelevant feature type,Not an activity-place feature type
